# Notebook 04: SAR vs. Optical Accuracy Comparison
## River Water Mask Study

**Researcher:** Bouchra Daddaoui  
**Supervisors:** Dr. Michael Nones, Dr. Kaveh Ghahraman  
**Institute:** Institute of Geophysics, Polish Academy of Sciences

---

### Objectives
1. Load pre-computed water masks (optical + SAR) and JRC reference
2. Compute accuracy metrics (F1, IoU, Kappa, Precision, Recall) per method per river
3. Statistical testing of method differences (Wilcoxon signed-rank)
4. Analyze performance by climate zone, river width, cloud cover, and season
5. Generate publication-quality figures


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sys.path.append('..')

from src.metrics import (
    compute_metrics, bootstrap_metrics, wilcoxon_test,
    temporal_consistency_index, compare_methods
)
from src.visualization import (
    plot_accuracy_comparison, plot_metrics_heatmap,
    plot_climate_performance, plot_mask_comparison
)
from src.data_loader import get_selected_rivers, RIVER_CONFIG

RESULTS_DIR = Path('../results')
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

%matplotlib inline
print('Libraries loaded.')

## 1. Load Water Masks and Reference Data

> **Note:** Masks are loaded from GeoTIFF exports from notebooks 02 and 03.
> Format: binary arrays (1=water, 0=non-water) as numpy arrays.

In [ ]:
import rasterio

rivers_df = get_selected_rivers()

def load_mask(path: str) -> np.ndarray:
    """Load a GeoTIFF binary mask as a numpy array."""
    with rasterio.open(path) as src:
        return src.read(1).astype(int)

# Example loading structure (replace with actual file paths after export)
# masks = {
#     'R01': {
#         'reference':  load_mask('processed/jrc/R01_JRC_20210601.tif'),
#         'MNDWI':      load_mask('processed/optical/R01_MNDWI_20210601.tif'),
#         'SAR_Otsu':   load_mask('processed/sar/R01_SAR_Otsu_20210601.tif'),
#         'NDWI':       load_mask('processed/optical/R01_NDWI_20210601.tif'),
#         ...
#     },
# }

print('Mask loading structure defined.')
print('Fill in actual paths after GEE export is complete.')

## 2. Compute Accuracy Metrics

In [ ]:
# Demonstration with synthetic data
# Replace with real masks after data processing
np.random.seed(42)
n_pixels = 10000

# Simulate reference and method predictions (for demonstration)
y_ref = np.random.choice([0, 1], size=n_pixels, p=[0.7, 0.3])

# Simulate method predictions with varying quality
def simulate_prediction(y_true, f1_target=0.85, seed=0):
    rng = np.random.default_rng(seed)
    noise = rng.random(len(y_true))
    pred = y_true.copy()
    flip_prob = 1 - f1_target
    pred[noise < flip_prob] = 1 - pred[noise < flip_prob]
    return pred

predictions = {
    'NDWI':       simulate_prediction(y_ref, 0.82, seed=1),
    'MNDWI':      simulate_prediction(y_ref, 0.87, seed=2),
    'AWEInsh':    simulate_prediction(y_ref, 0.85, seed=3),
    'SAR-Fixed':  simulate_prediction(y_ref, 0.78, seed=4),
    'SAR-Otsu':   simulate_prediction(y_ref, 0.83, seed=5),
    'SAR-Tile':   simulate_prediction(y_ref, 0.86, seed=6),
    'SAR-Change': simulate_prediction(y_ref, 0.91, seed=7),
}

# Compute metrics for all methods
all_metrics = {name: compute_metrics(y_ref, pred) for name, pred in predictions.items()}

# Display comparison
compare_methods(all_metrics)

# Convert to DataFrame for analysis
metrics_df = pd.DataFrame(all_metrics).T
display(metrics_df[['precision', 'recall', 'f1', 'iou', 'kappa', 'overall_accuracy']])

In [ ]:
# Bootstrapped confidence intervals for F1
ci_results = {}
for method_name, pred in predictions.items():
    ci = bootstrap_metrics(y_ref, pred, metric='f1', n_bootstrap=1000)
    ci_results[method_name] = ci
    print(f"{method_name:<15}: F1 = {ci['mean']:.3f} [{ci['lower']:.3f}–{ci['upper']:.3f}]")

# Plot with error bars
fig, ax = plt.subplots(figsize=(10, 5))
methods = list(ci_results.keys())
means = [ci_results[m]['mean'] for m in methods]
lowers = [ci_results[m]['mean'] - ci_results[m]['lower'] for m in methods]
uppers = [ci_results[m]['upper'] - ci_results[m]['mean'] for m in methods]

colors = ['#4CAF50' if 'SAR' not in m else '#9C27B0' for m in methods]
ax.bar(methods, means, yerr=[lowers, uppers], color=colors,
       capsize=5, edgecolor='white', linewidth=1.2)
ax.set_ylabel('F1 Score (95% CI)')
ax.set_title('Water Mask Accuracy by Method (Bootstrapped 95% CI)')
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'method_f1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Statistical Comparison: SAR vs. Optical

In [ ]:
# Wilcoxon signed-rank test: MNDWI vs. SAR-Otsu across rivers
# (Replace with actual per-river F1 scores after data processing)

# Example: F1 scores per river for each method
f1_mndwi_per_river   = [0.92, 0.71, 0.95, 0.78, 0.88, 0.90, 0.86, 0.83]
f1_sar_otsu_per_river = [0.85, 0.88, 0.90, 0.89, 0.87, 0.85, 0.82, 0.91]
river_names = ['Amazon', 'Congo', 'Nile', 'Ganges', 'Yangtze', 'Mississippi', 'Rhine', 'Ob']

test_result = wilcoxon_test(f1_mndwi_per_river, f1_sar_otsu_per_river)

print('Wilcoxon Signed-Rank Test: MNDWI vs. SAR-Otsu')
print(f"  Statistic: {test_result['statistic']}")
print(f"  p-value: {test_result['p_value']}")
print(f"  Significant (α=0.05): {test_result['significant']}")
print(f"  Direction: {test_result['direction']}")
print(f"  Mean F1 MNDWI: {test_result['mean_a']:.3f}")
print(f"  Mean F1 SAR:   {test_result['mean_b']:.3f}")

In [ ]:
# Per-river comparison scatter
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(f1_mndwi_per_river, f1_sar_otsu_per_river,
           s=120, zorder=5, c='#1565C0', edgecolors='black', linewidth=0.8)

for i, river in enumerate(river_names):
    ax.annotate(river, (f1_mndwi_per_river[i], f1_sar_otsu_per_river[i]),
                xytext=(6, 4), textcoords='offset points', fontsize=9)

ax.plot([0.6, 1.0], [0.6, 1.0], 'k--', alpha=0.4, label='Equal performance')
ax.set_xlabel('F1 Score — MNDWI (Optical)')
ax.set_ylabel('F1 Score — SAR Otsu')
ax.set_title('Per-River Performance: Optical vs. SAR\n(Above diagonal: SAR better)')
ax.set_xlim(0.65, 1.0)
ax.set_ylim(0.65, 1.0)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'optical_vs_sar_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Performance by Climate Zone

In [ ]:
climate_data = {
    'Amazon':      {'f1_optical': 0.71, 'f1_sar': 0.88, 'climate': 'Tropical Rainforest', 'cloud_pct': 82},
    'Congo':       {'f1_optical': 0.71, 'f1_sar': 0.88, 'climate': 'Tropical Wet-Dry',   'cloud_pct': 73},
    'Nile':        {'f1_optical': 0.95, 'f1_sar': 0.90, 'climate': 'Arid Desert',         'cloud_pct': 4},
    'Ganges':      {'f1_optical': 0.78, 'f1_sar': 0.89, 'climate': 'Tropical Monsoon',    'cloud_pct': 62},
    'Yangtze':     {'f1_optical': 0.88, 'f1_sar': 0.87, 'climate': 'Humid Subtropical',   'cloud_pct': 55},
    'Mississippi': {'f1_optical': 0.90, 'f1_sar': 0.85, 'climate': 'Humid Subtropical',   'cloud_pct': 45},
    'Rhine':       {'f1_optical': 0.86, 'f1_sar': 0.82, 'climate': 'Oceanic',              'cloud_pct': 62},
    'Ob':          {'f1_optical': 0.83, 'f1_sar': 0.91, 'climate': 'Subarctic',            'cloud_pct': 40},
}

cdf = pd.DataFrame(climate_data).T.reset_index()
cdf.columns = ['River', 'F1_Optical', 'F1_SAR', 'Climate', 'Cloud_Pct']
cdf[['F1_Optical', 'F1_SAR', 'Cloud_Pct']] = cdf[['F1_Optical', 'F1_SAR', 'Cloud_Pct']].astype(float)
cdf['SAR_Advantage'] = cdf['F1_SAR'] - cdf['F1_Optical']

# Cloud cover vs. optical F1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(cdf['Cloud_Pct'], cdf['F1_Optical'], s=100, c='#4CAF50', zorder=5, label='Optical F1')
axes[0].scatter(cdf['Cloud_Pct'], cdf['F1_SAR'], s=100, c='#9C27B0', zorder=5, marker='s', label='SAR F1')
for _, row in cdf.iterrows():
    axes[0].annotate(row['River'], (row['Cloud_Pct'], row['F1_Optical']), xytext=(3, 3),
                     textcoords='offset points', fontsize=8, color='#4CAF50')
axes[0].set_xlabel('Annual Cloud Cover (%)')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 Score vs. Cloud Cover')
axes[0].legend()

# SAR advantage by climate
cdf_sorted = cdf.sort_values('SAR_Advantage')
colors = ['#E53935' if v > 0 else '#4CAF50' for v in cdf_sorted['SAR_Advantage']]
axes[1].barh(cdf_sorted['River'], cdf_sorted['SAR_Advantage'], color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('F1 SAR − F1 Optical (positive = SAR better)')
axes[1].set_title('SAR Advantage by River')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'climate_performance.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Summary Results Table

In [ ]:
# Final summary table
summary = cdf[['River', 'Climate', 'Cloud_Pct', 'F1_Optical', 'F1_SAR', 'SAR_Advantage']].copy()
summary['Best_Method'] = summary['SAR_Advantage'].apply(
    lambda x: 'SAR' if x > 0.02 else 'Optical' if x < -0.02 else 'Comparable'
)
summary = summary.round(3)
display(summary.sort_values('SAR_Advantage', ascending=False))

# Save to CSV
summary.to_csv('../results/tables/method_comparison_summary.csv', index=False)
print('Results saved to results/tables/method_comparison_summary.csv')

## Key Findings

Update this section after running with real data:

1. **Overall**: [SAR/Optical/Comparable] method performs better across all rivers
2. **Tropical rivers**: SAR clearly outperforms optical (cloud cover)
3. **Arid rivers**: Optical performs better (clear skies; narrow channels)
4. **Cloud cover threshold**: Optical reliability drops below ~X% clear-sky availability
5. **Best SAR method**: [method] with F1 = X.XX
6. **Best optical method**: [method] with F1 = X.XX
7. **Wilcoxon test**: Methods are [significantly / not significantly] different (p = X.XXX)